In [21]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../data/clean/supply_chain.db")

In [22]:
pd.read_sql("""
SELECT COUNT(DISTINCT order_id) AS orders,
       COUNT(DISTINCT CASE WHEN is_late = 1 THEN order_id END) AS late_orders,
       ROUND(100.0 * COUNT(DISTINCT CASE WHEN is_late = 1 THEN order_id END)
             / COUNT(DISTINCT order_id), 1) AS late_pct
FROM orders
WHERE is_canceled = 0
""", conn)

,orders,late_orders,late_pct
0,62897,36048,57.3


In [23]:
pd.read_sql("""
SELECT shipping_mode,
       COUNT(DISTINCT order_id) AS orders,
       ROUND(100.0 * COUNT(DISTINCT CASE WHEN is_late = 1 THEN order_id END)
             / COUNT(DISTINCT order_id), 1) AS late_pct,
       ROUND(AVG(days_for_shipping_real), 2) AS avg_real_days,
       ROUND(AVG(days_for_shipment_scheduled), 2) AS avg_scheduled_days
FROM orders
WHERE is_canceled = 0
GROUP BY shipping_mode
ORDER BY late_pct DESC
""", conn)

,shipping_mode,orders,late_pct,avg_real_days,avg_scheduled_days
0,First Class,9602,100.0,2.00,1.0
1,Second Class,12256,80.0,3.99,2.0
2,Same Day,3407,48.4,0.48,0.0
3,Standard Class,37632,39.8,3.99,4.0


In [24]:
pd.read_sql("""
SELECT market,
       COUNT(DISTINCT order_id) AS orders,
       ROUND(100.0 * COUNT(DISTINCT CASE WHEN is_late = 1 THEN order_id END)
             / COUNT(DISTINCT order_id), 1) AS late_pct
FROM orders
WHERE is_canceled = 0
GROUP BY market
ORDER BY late_pct DESC
""", conn)

,market,orders,late_pct
0,Pacific Asia,16853,57.7
1,USCA,8178,57.5
2,Europe,17763,57.4
3,LATAM,16414,56.9
4,Africa,3689,56.5


In [25]:
pd.read_sql("""
SELECT order_region,
       COUNT(DISTINCT order_id) AS orders,
       ROUND(100.0 * COUNT(DISTINCT CASE WHEN is_late = 1 THEN order_id END)
             / COUNT(DISTINCT order_id), 1) AS late_pct
FROM orders
WHERE is_canceled = 0
GROUP BY order_region
ORDER BY late_pct DESC
LIMIT 5
""", conn)

,order_region,orders,late_pct
0,Central Africa,533,60.0
1,East Africa,589,59.1
2,South of USA,1280,58.8
3,West Asia,1935,58.5
4,Western Europe,9563,58.4


In [26]:
pd.read_sql("""
SELECT market,
       COUNT(DISTINCT order_id) AS orders,
       ROUND(SUM(sales), 0) AS total_sales,
       ROUND(SUM(order_profit_per_order), 0) AS total_profit,
       ROUND(100.0 * SUM(order_profit_per_order) / SUM(sales), 1) AS profit_margin_pct
FROM orders
WHERE is_canceled = 0
GROUP BY market
ORDER BY total_profit DESC
""", conn)

,market,orders,total_sales,total_profit,profit_margin_pct
0,Europe,17763,10405371.0,1131938.0,10.9
1,LATAM,16414,9824330.0,1069051.0,10.9
2,Pacific Asia,16853,7942352.0,826167.0,10.4
3,USCA,8178,4836414.0,539223.0,11.1
4,Africa,3689,2205963.0,240041.0,10.9


In [27]:
pd.read_sql("""
SELECT category_name,
       ROUND(SUM(sales), 0) AS total_sales,
       ROUND(SUM(order_profit_per_order), 0) AS total_profit,
       ROUND(100.0 * SUM(order_profit_per_order) / SUM(sales), 1) AS margin_pct
FROM orders
WHERE is_canceled = 0
GROUP BY category_name
ORDER BY total_profit DESC
""", conn)

,category_name,total_sales,total_profit,margin_pct
0,Fishing,6637668.0,731576.0,11.0
1,Cleats,4244594.0,474630.0,11.2
2,Camping & Hiking,3946837.0,409896.0,10.4
3,Cardio Equipment,3533289.0,362985.0,10.3
4,Women's Apparel,3011600.0,334336.0,11.1
5,Water Sports,2981851.0,308981.0,10.4
6,Indoor/Outdoor Games,2765543.0,302404.0,10.9
7,Men's Footwear,2763977.0,296711.0,10.7
8,Shop By Sport,1253396.0,125077.0,10.0
9,Computers,637500.0,69232.0,10.9


In [28]:
pd.read_sql("""
SELECT is_late,
       COUNT(*) AS items,
       ROUND(AVG(order_profit_per_order), 2) AS avg_profit_per_item,
       ROUND(100.0 * SUM(order_profit_per_order) / SUM(sales), 1) AS margin_pct
FROM orders
WHERE is_canceled = 0
GROUP BY is_late
""", conn)

,is_late,items,avg_profit_per_item,margin_pct
0,0,73788,22.58,11.0
1,1,98977,21.62,10.6


In [29]:
abc = pd.read_sql("""
WITH cat AS (
    SELECT category_name,
           SUM(sales) AS total_sales,
           SUM(order_profit_per_order) AS total_profit
    FROM orders
    WHERE is_canceled = 0
    GROUP BY category_name
),
ranked AS (
    SELECT *,
           SUM(total_profit) OVER (ORDER BY total_profit DESC) AS running_profit,
           SUM(total_profit) OVER () AS grand_profit
    FROM cat
)
SELECT category_name,
       ROUND(total_sales, 0) AS total_sales,
       ROUND(total_profit, 0) AS total_profit,
       ROUND(100.0 * running_profit / grand_profit, 1) AS cumulative_pct,
       CASE WHEN 100.0 * (running_profit - total_profit) / grand_profit < 80 THEN 'A'
            WHEN 100.0 * (running_profit - total_profit) / grand_profit < 95 THEN 'B'
            ELSE 'C' END AS abc_class
FROM ranked
ORDER BY total_profit DESC
""", conn)

abc.groupby("abc_class").agg(categories=("category_name", "count"),
                             profit=("total_profit", "sum"))

,categories,profit
abc_class,,
A,8,3221519.0
B,10,398771.0
C,32,186131.0


In [30]:
abc = pd.read_sql("""
WITH cat AS (
    SELECT category_name,
           SUM(sales) AS total_sales,
           SUM(order_profit_per_order) AS total_profit
    FROM orders
    WHERE is_canceled = 0
    GROUP BY category_name
),
ranked AS (
    SELECT *,
           SUM(total_profit) OVER (ORDER BY total_profit DESC) AS running_profit,
           SUM(total_profit) OVER () AS grand_profit
    FROM cat
)
SELECT category_name,
       ROUND(total_sales, 0) AS total_sales,
       ROUND(total_profit, 0) AS total_profit,
       ROUND(100.0 * running_profit / grand_profit, 1) AS cumulative_pct,
       CASE WHEN 100.0 * (running_profit - total_profit) / grand_profit < 80 THEN 'A'
            WHEN 100.0 * (running_profit - total_profit) / grand_profit < 95 THEN 'B'
            ELSE 'C' END AS abc_class
FROM ranked
ORDER BY total_profit DESC
""", conn)

abc.groupby("abc_class").agg(categories=("category_name", "count"),
                             profit=("total_profit", "sum"))

,categories,profit
abc_class,,
A,8,3221519.0
B,10,398771.0
C,32,186131.0


In [31]:
dash = pd.read_sql("""
SELECT order_id, order_date, order_year, order_month,
       shipping_mode, delivery_status, is_late, delay_days,
       days_for_shipping_real, days_for_shipment_scheduled,
       market, order_region, order_country,
       category_name, product_name, customer_segment,
       order_item_quantity, sales, order_profit_per_order
FROM orders
WHERE is_canceled = 0
""", conn)

dash = dash.merge(abc[["category_name", "abc_class"]], on="category_name", how="left")

print(dash.shape)
print(dash["abc_class"].isna().sum())
dash.to_csv("../data/clean/dashboard_data.csv", index=False)

(172765, 20)
0
